<a href="https://colab.research.google.com/github/RunyuZhu-Hub/Undergraduate/blob/main/Activity_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Activity 4: Gram-Schmidt and the QR factorization
## Reminders
In this exercise, we'll be writing code to transform arbitrary bases of $\mathbb{R}^n$ into orthogonal bases with the Gram-Schmidt process.
We'll start by reviewing some built-in functions that will be helpful.

In [1]:
import numpy as np
from numpy import linalg as LA
from numpy import random as RA

Recall that slices allow us to work directly with columns and (in most ways) treat them as vectors. Given a numpy `array` `A`, we can extract its columns using slicing. Here are a few quick examples.

In [2]:
A=np.array([[2., 1., 0., 0.],
       [3., 2., 1., 0.],
       [0., 3., 2., 1.],
       [0., 0., 3., 2.]])
A[:,0] #first column

array([2., 3., 0., 0.])

Note that slicing gives us a *view* rather than a *copy*. If what we really want is a copy, we can call the `copy()` method

In [3]:
A[:,0]=A[:,0]*(-2) #multiply the first column by -2
A #Now note that indeed the first column is altered

array([[-4.,  1.,  0.,  0.],
       [-6.,  2.,  1.,  0.],
       [-0.,  3.,  2.,  1.],
       [-0.,  0.,  3.,  2.]])

In [4]:
v=A[:,1].copy() #call a copy of the second column
v=v*5 #multiply v by 5
A,v #check if A has changed (and compare to v)

(array([[-4.,  1.,  0.,  0.],
        [-6.,  2.,  1.,  0.],
        [-0.,  3.,  2.,  1.],
        [-0.,  0.,  3.,  2.]]),
 array([ 5., 10., 15.,  0.]))

## Exercise 1: The Gram-Schmidt process
Now, we'll implement the Gram-Schmidt process.
We'll take the columns of an input matrix to be the basis we start with.
That is, in the notation of section 4.2 of Olver-Shakiban, we'll take the columns `A[0]`, `A[1]`, ... , `A[n-1]` of our `A` to be $\vec{w}_1,\dots, \vec{w}_n$.
Our output will be a new matrix `Q` with columns given by the Gram-Schmidt process. In the notation of section 4.2, these are $\vec{v}_1,\dots,\vec{v}_n$.

For our first pass of the Gram-Schmidt process, write a function which implements the Gram-Schmidt process for a  matrix `A` with linearly independent columns using the Gram-Schmidt formula.
You may do this however you like, but an outline is given in the hint below.
<details>
    <summary> <b> Hint: </b> (Click here to expand) </summary>
    
* Set `n` to be the number of *columns* in `A`.
* Initialize a new matrix `Q` in the shape of `A`.
* Iterate over `j` in `range(n)` parametrizing which column we are working with.
* Adapt the formula $$\vec{v}_j=\vec{w}_j-\sum_{i=1}^{j-1} \frac{\langle \vec{w}_j,\vec{v_i}\rangle}{\langle \vec{v}_i,\vec{v}_i\rangle}\vec{v}_i$$ to the naming and indexing established and use it to set the column `Q[j]`.
* Return the matrix `Q`.
    </details>

In [11]:
def my_GS(A):
    n = A.shape[1]
    Q = np.zeros_like(A)

    for j in range(n):
        v_j = A[:, j].copy()

        for i in range(j):
            v_i = Q[:, i]

            projection = (np.dot(A[:, j], v_i) / np.dot(v_i, v_i)) * v_i
            v_j -= projection

        Q[:, j] = v_j

    return Q

In [12]:
#testing
A1=np.array([[1,2,3],
            [4,5,6],
            [1,-1,1],
            [0,-2,-1]],"float64")
Q=my_GS(A)
Q,np.dot(np.transpose(Q),Q)  #Question: what's special about this product?
#Desired output:
#(array([[ 1.        ,  0.83333333,  1.06432749],
#       [ 4.        ,  0.33333333, -0.37426901],
#       [ 1.        , -2.16666667,  0.43274854],
#       [ 0.        , -2.        , -0.0877193 ]]),
# array([[ 1.80000000e+01, -1.77635684e-15, -2.22044605e-16],
#        [-1.77635684e-15,  9.50000000e+00,  1.52655666e-15],
#        [-2.22044605e-16,  1.52655666e-15,  1.46783626e+00]]))

(array([[-4.        , -0.23076923, -0.30508475,  0.2755102 ],
        [-6.        ,  0.15384615,  0.20338983, -0.18367347],
        [-0.        ,  3.        , -0.03389831,  0.03061224],
        [-0.        ,  0.        ,  3.        ,  0.04081633]]),
 array([[ 5.20000000e+01,  8.88178420e-16, -7.21644966e-16,
          3.33066907e-16],
        [ 8.88178420e-16,  9.07692308e+00,  2.77555756e-16,
         -2.46330734e-16],
        [-7.21644966e-16,  2.77555756e-16,  9.13559322e+00,
         -5.27355937e-16],
        [ 3.33066907e-16, -2.46330734e-16, -5.27355937e-16,
          1.12244898e-01]]))

Now, copy and modify your code to give an orthonormal basis. That is, after finding each $\vec{v}_i$, set $\vec u_i=\vec v_i/\lVert{\vec v_i}\rVert$ and let the output matrix's columns `Q[0]`,`Q[1]`,...,`Q[n-1]` be $\vec u_1,\dots , \vec U_n$.

You'll likely find the built-in `numpy` command `np.linalg.norm` useful, imported for convenience as `LA.norm`.

In [21]:
def my_GS_normalized(A):
    n = A.shape[1]
    Q = np.zeros_like(A, dtype=np.float64)  # Initialize Q with float type

    for j in range(n):
        v_j = A[:, j].copy()

        for i in range(j):
            v_i = Q[:, i]
            # Check for zero vector before calculating projection
            if np.dot(v_i, v_i) > 1e-10: # Add a tolerance for numerical stability
                projection = (np.dot(A[:, j], v_i) / np.dot(v_i, v_i)) * v_i
                v_j -= projection
            else:
                # Handle the case where v_i is a zero vector (linearly dependent)
                # For now, we can skip the projection as there's nothing to project onto
                pass


        # Normalize the vector v_j
        norm_v_j = LA.norm(v_j)
        if norm_v_j > 1e-10: # Add a tolerance for numerical stability
            Q[:, j] = v_j / norm_v_j
        else:
            # Handle the case where v_j is a zero vector (linearly dependent)
            # This indicates a linearly dependent input, but we will handle this
            # more formally in the next exercise. For now, we can just put a zero vector.
             Q[:, j] = np.zeros_like(v_j)

    return Q

In [22]:
#testing
A=np.array([[1,2,3],
            [4,5,6],
            [1,-1,1],
            [0,-2,-1]],"float64")
Q=my_GS_normalized(A)
Q,np.dot(np.transpose(Q),Q)
#Desired output:
#(array([[ 0.23570226,  0.27036904,  0.87848929],
#        [ 0.94280904,  0.10814761, -0.30891931],
#        [ 0.23570226, -0.70295949,  0.35718795],
#        [ 0.        , -0.64888568, -0.07240296]]),
# array([[ 1.00000000e+00, -1.36129200e-16, -7.70383744e-16],
#        [-1.36129200e-16,  1.00000000e+00,  4.73463928e-16],
#        [-7.70383744e-16,  4.73463928e-16,  1.00000000e+00]]))

(array([[ 0.23570226,  0.27036904,  0.87848929],
        [ 0.94280904,  0.10814761, -0.30891931],
        [ 0.23570226, -0.70295949,  0.35718795],
        [ 0.        , -0.64888568, -0.07240296]]),
 array([[ 1.00000000e+00,  2.21518547e-16,  4.69478111e-16],
        [ 2.21518547e-16,  1.00000000e+00, -1.43599196e-15],
        [ 4.69478111e-16, -1.43599196e-15,  1.00000000e+00]]))

## Exercise 2: Error Handling
Modify your functions to `raise` an `exception` when a linearly dependent set is given as input.

There is one major complication to this exercise: often, numpy returns a value on the order of `10**-15` or so where mathematically we should expect zero due to rounding errors. To account for this, we should take an additional parameter `tolerance` (which is set by default to `10**(-10)`) such that we ignore any variation of magnitude less than `tolerance`. (how can you make that instruction precise?)

In [26]:
def my_GS_safe(A, tolerance=1e-10):
    n = A.shape[1]
    Q = np.zeros_like(A)

    for j in range(n):
        v_j = A[:, j].copy()

        for i in range(j):
            v_i = Q[:, i]
            # Check for near-zero norm before calculating projection
            if np.dot(v_i, v_i) < tolerance:
                 # If v_i is close to zero, the input vectors are linearly dependent
                 raise Exception("Linearly dependent input")

            projection = (np.dot(A[:, j], v_i) / np.dot(v_i, v_i)) * v_i
            v_j -= projection

        # Check if the resulting vector is close to zero
        if np.dot(v_j, v_j) < tolerance:
             # If v_j is close to zero, the input vectors are linearly dependent
             raise Exception("Linearly dependent input")

        Q[:, j] = v_j

    return Q

In [28]:
#testing
A1=np.array([[1,2,3],
            [4,5,6],
            [1,-1,1],
            [0,-2,-1]],"float64")
A2=np.array([[1,2,3,0],
            [4,5,6,3],
            [1,-1,1,-1],
            [0,-2,-1,-1]],"float64")
P1=my_GS_safe(A1)
print(P1)
P2=my_GS_safe(A2)
print(P2)
#Desired Output:
#[[ 1.          0.83333333  1.06432749]
# [ 4.          0.33333333 -0.37426901]
# [ 1.         -2.16666667  0.43274854]
# [ 0.         -2.         -0.0877193 ]]
#---------------------------------------------------------------------------
#Exception
# ...
# Linearly dependent input

[[ 1.          0.83333333  1.06432749]
 [ 4.          0.33333333 -0.37426901]
 [ 1.         -2.16666667  0.43274854]
 [ 0.         -2.         -0.0877193 ]]


Exception: Linearly dependent input

In [29]:
#more testing
A1=np.array([[1,2,3],
            [4,5,6],
            [1,-1,1],
            [0,-2,-1]],"float64")
A2=np.array([[1,2,3,0],
            [4,5,6,3],
            [1,-1,1,-1],
            [0,-2,-1,-1]],"float64")
Q1=my_GS_normalized_safe(A1)
print(Q1)
Q2=my_GS_normalized_safe(A2)
print(Q2)
#Desired output:
#[[ 0.23570226  0.27036904  0.87848929]
# [ 0.94280904  0.10814761 -0.30891931]
# [ 0.23570226 -0.70295949  0.35718795]
# [ 0.         -0.64888568 -0.07240296]]
#
#---------------------------------------------------------------------------
#Exception
#...
#Exception: Linearly dependent input

NameError: name 'my_GS_normalized_safe' is not defined

In [33]:
def my_GS_normalized_safe(A, tolerance=1e-10):
    n = A.shape[1]
    Q = np.zeros_like(A, dtype=np.float64)

    for j in range(n):
        v_j = A[:, j].copy()

        for i in range(j):
            v_i = Q[:, i]
            # Check for near-zero norm before calculating projection
            if np.dot(v_i, v_i) < tolerance:
                 # If v_i is close to zero, the input vectors are linearly dependent
                 raise Exception("Linearly dependent input")

            projection = (np.dot(A[:, j], v_i) / np.dot(v_i, v_i)) * v_i
            v_j -= projection

        # Check if the resulting vector is close to zero before normalizing
        norm_v_j = LA.norm(v_j)
        if norm_v_j < tolerance:
             # If v_j is close to zero, the input vectors are linearly dependent
             raise Exception("Linearly dependent input")

        # Normalize the vector v_j
        Q[:, j] = v_j / norm_v_j

    return Q

## Exercise 3: The QR Factorization

Now, we want to modify our code once more to produce a QR factorization of a square input matrix. Note that `my_GS_normalized` already produces an orthogonal matrix `Q` as its output given square nonsingular input.

**Exercise:** Modify `my_GS_normalized` to produce a QR factorization of a square input matrix `A` (or work from scratch!). The process is described on pages 197-8 of Olver-Shakiban, but you may find the "numerically stable" version on page 199 easier to implement. Be sure to check your code with tests--we should have that $A=QR$ and that $QQ^T=I$

In [37]:
def my_QR(A):
    n = A.shape[1]
    Q = np.zeros_like(A, dtype=np.float64)
    R = np.zeros((n, n), dtype=np.float64)

    for j in range(n):
        v_j = A[:, j].copy()
        for i in range(j):
            v_i = Q[:, i]
            R[i, j] = np.dot(A[:, j], v_i)
            v_j -= R[i, j] * v_i

        R[j, j] = LA.norm(v_j)
        Q[:, j] = v_j / R[j, j]

    return Q, R

In [38]:
#testing
A3=np.array([[1,2,3],
            [4,5,6],
            [1,-1,1]],"float64")
Q3,R3=my_QR(A3)
A3,Q3,R3,np.dot(Q3,R3),np.dot(np.transpose(Q3),Q3)
#Desired output:
#(array([[ 1.,  2.,  3.],
#        [ 4.,  5.,  6.],
#        [ 1., -1.,  1.]]),
# array([[ 0.23570226,  0.35533453,  0.90453403],
#        [ 0.94280904,  0.14213381, -0.30151134],
#        [ 0.23570226, -0.92386977,  0.30151134]]),
# array([[4.24264069, 4.94974747, 6.59966329],
#        [0.        , 2.34520788, 0.99493668],
#        [0.        , 0.        , 1.20604538]]),
# array([[ 1.,  2.,  3.],
#        [ 4.,  5.,  6.],
#        [ 1., -1.,  1.]]),
# array([[ 1.00000000e+00, -5.25448946e-16, -4.12858351e-16],
#        [-5.25448946e-16,  1.00000000e+00, -9.74257535e-18],
#        [-4.12858351e-16, -9.74257535e-18,  1.00000000e+00]]))

(array([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 1., -1.,  1.]]),
 array([[ 0.23570226,  0.35533453,  0.90453403],
        [ 0.94280904,  0.14213381, -0.30151134],
        [ 0.23570226, -0.92386977,  0.30151134]]),
 array([[4.24264069, 4.94974747, 6.59966329],
        [0.        , 2.34520788, 0.99493668],
        [0.        , 0.        , 1.20604538]]),
 array([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 1., -1.,  1.]]),
 array([[ 1.00000000e+00, -1.64626463e-16, -7.35607367e-16],
        [-1.64626463e-16,  1.00000000e+00,  6.56963394e-16],
        [-7.35607367e-16,  6.56963394e-16,  1.00000000e+00]]))

In [41]:
A3=np.array([[1,2,3],
            [4,5,6],
            [1,-1,1]],"float64")
Q3,R3=my_QR2(A3)
A3,Q3,R3,np.dot(Q3,R3),np.dot(np.transpose(Q3),Q3)
#Desired output:
#(array([[ 1.,  2.,  3.],
#        [ 4.,  5.,  6.],
#        [ 1., -1.,  1.]]),
# array([[ 0.23570226,  0.35533453,  0.90453403],
#        [ 0.94280904,  0.14213381, -0.30151134],
#        [ 0.23570226, -0.92386977,  0.30151134]]),
# array([[4.24264069, 4.94974747, 6.59966329],
#        [0.        , 2.34520788, 0.99493668],
#        [0.        , 0.        , 1.20604538]]),
# array([[ 1.,  2.,  3.],
#        [ 4.,  5.,  6.],
#        [ 1., -1.,  1.]]),
# array([[ 1.00000000e+00, -5.25448946e-16, -4.12858351e-16],
#        [-5.25448946e-16,  1.00000000e+00, -9.74257535e-18],
#        [-4.12858351e-16, -9.74257535e-18,  1.00000000e+00]]))

(array([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 1., -1.,  1.]]),
 array([[ 0.23570226,  0.35533453,  0.90453403],
        [ 0.94280904,  0.14213381, -0.30151134],
        [ 0.23570226, -0.92386977,  0.30151134]]),
 array([[4.24264069, 4.94974747, 6.59966329],
        [0.        , 2.34520788, 0.99493668],
        [0.        , 0.        , 1.20604538]]),
 array([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 1., -1.,  1.]]),
 array([[ 1.00000000e+00, -1.64626463e-16, -7.45131575e-16],
        [-1.64626463e-16,  1.00000000e+00, -1.39686520e-17],
        [-7.45131575e-16, -1.39686520e-17,  1.00000000e+00]]))

In [42]:
def my_QR2(A):
    m, n = A.shape
    Q = np.zeros((m, n), dtype=np.float64)
    R = np.zeros((n, n), dtype=np.float64)
    V = A.copy().astype(np.float64) # Use a copy to avoid modifying the original matrix

    for i in range(n):
        R[i, i] = LA.norm(V[:, i])
        Q[:, i] = V[:, i] / R[i, i]
        for j in range(i + 1, n):
            R[i, j] = np.dot(Q[:, i], V[:, j])
            V[:, j] -= R[i, j] * Q[:, i]

    return Q, R

## Exercise 4: Orthogonal Projection
Recall from Olver-Shakiban:
**Definition 4.31.** The *orthogonal projection* of $\vec v$ onto the subspace $W$ is the element $\vec w \in W$ such that $\vec z = \vec v - \vec w$ is orthogonal to $W$.

Note that Theorem 4.32 below it gives a formula for $\vec w$ in terms of an orthonormal basis for $W$.

**Exercise** Given a basis for $W$ (in the form of the columns of an array `A`) and a vector `v`, write a function `my_projection` which outputs the orthogonal projection `w` of `v` onto `W`.

In [46]:
def my_projection(A, v):
    # Get an orthonormal basis for the column space of A
    Q = my_GS_normalized(A)

    # Initialize the projection vector
    w = np.zeros_like(v, dtype=np.float64)

    # Calculate the orthogonal projection using the formula
    # w = sum(dot(v, u_i) * u_i) for u_i in the orthonormal basis Q
    for i in range(Q.shape[1]):
        u_i = Q[:, i]
        w += np.dot(v, u_i) * u_i

    return w

In [47]:
#testing
A1=np.array([[1,2,3],
            [4,5,6],
            [1,-1,1],
            [0,-2,-1]],"float64")
v=np.array([1,0,0,0],"float64")
w=my_projection(A1,v)
z=v-w
w,z,np.dot(np.transpose(A1),z)
#Desired output:
#(array([ 0.90039841, -0.01992032,  0.17928287, -0.23904382]),
# array([ 0.09960159,  0.01992032, -0.17928287,  0.23904382]),
# array([2.72004641e-15, 1.83186799e-15, 3.83026943e-15]))
#(or something very close to zero for the third line)

(array([ 0.90039841, -0.01992032,  0.17928287, -0.23904382]),
 array([ 0.09960159,  0.01992032, -0.17928287,  0.23904382]),
 array([-2.05391260e-15,  1.22124533e-15, -9.99200722e-16]))

## Exercise 5 (Challenge!): Comparing the approaches
In exercise 2, two QR algorithms were suggested to you, one on pages 197-98 and one on page 199 of Olver-Shakiban.

**(a)** Whichever one you did *not* implement, implement that one below as `my_QR2`.

In [48]:
def my_QR2():
    #YOUR CODE HERE
    return

**(b)** Write a function which measures failure of orthogonality of an input matrix $Q$ by giving the greatest entry of $\left\lvert Q^TQ-I\right \rvert$ where $I$ is the appropriately-sized identity matrix.

*Note:* You might find issues if you try to use the base python built-in `max`. There are workarounds for this, but the best solution is probably the `numpy`-provided `np.max`.

In [54]:
def orth_checker(Q):
    # Calculate Q^T * Q
    QTQ = np.dot(np.transpose(Q), Q)
    # Get the identity matrix of the same size
    I = np.eye(Q.shape[1])
    # Calculate the absolute difference between Q^T * Q and I
    diff = np.abs(QTQ - I)
    # Return the greatest entry of the difference
    return np.max(diff)

In [55]:
M=RA.rand(5,5)
(Q,R)=my_QR(M)
orth_checker(Q)

np.float64(1.0126353054523724e-15)

**(c):** `np.random.rand(m,n)` (imported for convenience as `RA.rand(m,n)`) returns a random matrix of dimensions $m\times n$ with entries in the interval $[0,1)$. Use this to write a function which takes as input a dimension `n` and a number of iterations `num_iters` and compares your two implementations by generating a random $n\times n$ matrix, using both QR algorithms to compute a factorization, and checking which returns a $Q$ closer to being on-the-nose orthogonal, doing this `num_iters` times. When run, your function should print a line for each test stating which iteration the test is on, which implementation gave a result with better `orth_checker` result, and that `orth_checker` result. Finally, when the test is concluded, print out how many times each algorithm 'won' in human-readable form.

In [58]:
def QR_comp(n, num_iters):
    my_qr_wins = 0
    my_qr2_wins = 0

    for i in range(num_iters):
        M = RA.rand(n, n)

        try:
            Q1, R1 = my_QR(M)
            ortho_error1 = orth_checker(Q1)
        except Exception as e:
            ortho_error1 = float('inf') # Assign a very large error if QR fails

        try:
            Q2, R2 = my_QR2(M)
            ortho_error2 = orth_checker(Q2)
        except Exception as e:
            ortho_error2 = float('inf') # Assign a very large error if QR2 fails

        print(f"Iteration {i+1}:")
        if ortho_error1 < ortho_error2:
            my_qr_wins += 1
            print(f"  my_QR had a better orthogonality check result: {ortho_error1}")
        elif ortho_error2 < ortho_error1:
            my_qr2_wins += 1
            print(f"  my_QR2 had a better orthogonality check result: {ortho_error2}")
        else:
            print(f"  Both algorithms had similar orthogonality check results: {ortho_error1}")

    print("\nComparison Results:")
    print(f"my_QR won: {my_qr_wins} times")
    print(f"my_QR2 won: {my_qr2_wins} times")

In [59]:
QR_comp(2, 80)

Iteration 1:
  my_QR had a better orthogonality check result: 2.220446049250313e-16
Iteration 2:
  my_QR had a better orthogonality check result: 1.1102230246251565e-16
Iteration 3:
  my_QR had a better orthogonality check result: 1.6672135216584602e-16
Iteration 4:
  my_QR had a better orthogonality check result: 2.220446049250313e-16
Iteration 5:
  my_QR had a better orthogonality check result: 6.489872489303537e-16
Iteration 6:
  my_QR had a better orthogonality check result: 1.1102230246251565e-16
Iteration 7:
  my_QR had a better orthogonality check result: 1.1102230246251565e-16
Iteration 8:
  my_QR had a better orthogonality check result: 4.1096464777926956e-16
Iteration 9:
  my_QR had a better orthogonality check result: 7.888390918178699e-16
Iteration 10:
  my_QR had a better orthogonality check result: 1.983093117618702e-16
Iteration 11:
  my_QR had a better orthogonality check result: 1.4181890812150517e-15
Iteration 12:
  my_QR had a better orthogonality check result: 2.220

(d) Try your comparison function with varying `n` and a large `num_iters`. Is one consistently better than the other? Do the results change in low or high dimension? Write a few words below. You might consider modifying your code to track how much closer to orthogonality the preferred one is compared to the other, perhaps computing the largest and average discrepancies.

(This is an empty markdown cell for you to type out your response)

In [62]:
def QR_diff(n, num_iters):
    my_qr_better_count = 0
    my_qr2_better_count = 0
    total_discrepancy = 0

    for i in range(num_iters):
        M = RA.rand(n, n)

        try:
            Q1, R1 = my_QR(M)
            ortho_error1 = orth_checker(Q1)
        except Exception as e:
            ortho_error1 = float('inf')

        try:
            Q2, R2 = my_QR2(M)
            ortho_error2 = orth_checker(Q2)
        except Exception as e:
            ortho_error2 = float('inf')

        print(f"Iteration {i+1} (n={n}):")
        if ortho_error1 < ortho_error2:
            my_qr_better_count += 1
            print(f"  my_QR had a better orthogonality check result: {ortho_error1}")
            total_discrepancy += (ortho_error2 - ortho_error1)
        elif ortho_error2 < ortho_error1:
            my_qr2_better_count += 1
            print(f"  my_QR2 had a better orthogonality check result: {ortho_error2}")
            total_discrepancy += (ortho_error1 - ortho_error2)
        else:
            print(f"  Both algorithms had similar orthogonality check results: {ortho_error1}")


    print(f"\nComparison Results for n={n} ({num_iters} iterations):")
    print(f"my_QR was better: {my_qr_better_count} times")
    print(f"my_QR2 was better: {my_qr2_better_count} times")
    if my_qr_better_count > my_qr2_better_count:
        print(f"On average, my_QR was better by approximately: {total_discrepancy / my_qr_better_count}")
    elif my_qr2_better_count > my_qr_better_count:
         print(f"On average, my_QR2 was better by approximately: {total_discrepancy / my_qr2_better_count}")
    else:
        print("The two algorithms performed similarly in terms of average discrepancy.")

In [63]:
QR_diff(2,1000)

Iteration 1 (n=2):
  my_QR had a better orthogonality check result: 1.5315152090401414e-16
Iteration 2 (n=2):
  my_QR had a better orthogonality check result: 1.1102230246251565e-16
Iteration 3 (n=2):
  my_QR had a better orthogonality check result: 1.0653874051008442e-15
Iteration 4 (n=2):
  my_QR had a better orthogonality check result: 7.417121992542817e-17
Iteration 5 (n=2):
  my_QR had a better orthogonality check result: 4.554907204691363e-16
Iteration 6 (n=2):
  my_QR had a better orthogonality check result: 8.1715481082739735e-16
Iteration 7 (n=2):
  my_QR had a better orthogonality check result: 2.220446049250313e-16
Iteration 8 (n=2):
  my_QR had a better orthogonality check result: 3.983414175527772e-16
Iteration 9 (n=2):
  my_QR had a better orthogonality check result: 3.3306690738754696e-16
Iteration 10 (n=2):
  my_QR had a better orthogonality check result: 2.220446049250313e-16
Iteration 11 (n=2):
  my_QR had a better orthogonality check result: 3.120364763586694e-16
Ite

In [64]:
QR_diff(10, 1000)

Iteration 1 (n=10):
  my_QR had a better orthogonality check result: 3.0421115092614384e-14
Iteration 2 (n=10):
  my_QR had a better orthogonality check result: 1.9913323322452614e-14
Iteration 3 (n=10):
  my_QR had a better orthogonality check result: 4.90041925740062e-15
Iteration 4 (n=10):
  my_QR had a better orthogonality check result: 3.3289816567639675e-13
Iteration 5 (n=10):
  my_QR had a better orthogonality check result: 2.6795388340306556e-15
Iteration 6 (n=10):
  my_QR had a better orthogonality check result: 1.2391266273819428e-14
Iteration 7 (n=10):
  my_QR had a better orthogonality check result: 1.4410550694786093e-13
Iteration 8 (n=10):
  my_QR had a better orthogonality check result: 1.1858946311124138e-15
Iteration 9 (n=10):
  my_QR had a better orthogonality check result: 3.802695816256712e-15
Iteration 10 (n=10):
  my_QR had a better orthogonality check result: 5.193608794156406e-15
Iteration 11 (n=10):
  my_QR had a better orthogonality check result: 1.66509164390

In [65]:
QR_diff(100, 100)

Iteration 1 (n=100):
  my_QR had a better orthogonality check result: 1.0152322925585856e-11
Iteration 2 (n=100):
  my_QR had a better orthogonality check result: 1.4687484045081334e-12
Iteration 3 (n=100):
  my_QR had a better orthogonality check result: 1.0228203719962482e-11
Iteration 4 (n=100):
  my_QR had a better orthogonality check result: 3.286203523161295e-13
Iteration 5 (n=100):
  my_QR had a better orthogonality check result: 5.214812746723634e-13
Iteration 6 (n=100):
  my_QR had a better orthogonality check result: 1.0903081589685785e-12
Iteration 7 (n=100):
  my_QR had a better orthogonality check result: 4.7224075730247e-13
Iteration 8 (n=100):
  my_QR had a better orthogonality check result: 7.278381239318153e-13
Iteration 9 (n=100):
  my_QR had a better orthogonality check result: 4.635398633668798e-13
Iteration 10 (n=100):
  my_QR had a better orthogonality check result: 3.565942762008115e-12
Iteration 11 (n=100):
  my_QR had a better orthogonality check result: 1.3528